First experiment

In [3]:
#!pip install matplotlib

In [4]:
"""Reproducible split-SDE solver experiments (NumPy + Matplotlib)."""
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20261010)


def gbm(nsteps, paths=20000, mu=-0.4, sigma=0.7, T=1.0):
    """Use identical Brownian increments for all solvers and exact reference."""
    h = T / nsteps
    dW = rng.normal(0.0, np.sqrt(h), size=(paths, nsteps))
    wT = dW.sum(axis=1)
    exact = np.exp((mu - 0.5 * sigma**2) * T + sigma * wT)
    em = np.ones(paths)
    lie = np.ones(paths)
    strang = np.ones(paths)
    for dw in dW.T:
        em *= 1 + mu*h + sigma*dw
        lie *= np.exp(mu*h) * np.exp(-0.5*sigma**2*h + sigma*dw)
        strang *= np.exp(mu*h/2) * np.exp(-0.5*sigma**2*h + sigma*dw) * np.exp(mu*h/2)
    return {name: (np.mean((x-exact)**2)**0.5, abs(np.mean(x)-np.exp(mu*T)))
            for name, x in [('Euler-Maruyama', em), ('Lie', lie), ('Strang', strang)]}


def ou(nsteps, paths=100000, lam=3.0, m=1.0, sigma=0.8, T=2.0):
    """Compare exact OU moments with drift/noise split approximations."""
    h = T / nsteps
    exact_mean = m * (1 - np.exp(-lam*T))
    exact_var = sigma**2/(2*lam) * (1-np.exp(-2*lam*T))
    lie = np.zeros(paths)
    strang = np.zeros(paths)
    for _ in range(nsteps):
        dw = rng.normal(0, np.sqrt(h), size=paths)
        lie = m + (lie-m)*np.exp(-lam*h) + sigma*dw
        strang = m + ((m + (strang-m)*np.exp(-lam*h/2) + sigma*dw)-m)*np.exp(-lam*h/2)
    return {name: (abs(x.mean()-exact_mean), abs(x.var()-exact_var))
            for name, x in [('Lie', lie), ('Strang', strang)]}


if __name__ == '__main__':
    grids = [8, 16, 32, 64, 128, 256]
    gbm_results = [gbm(n) for n in grids]
    ou_results = [ou(n) for n in grids]
    print('GBM: strong RMSE, weak mean bias')
    for n, result in zip(grids, gbm_results):
        print(n, result)
    print('\nOU: mean bias, variance bias')
    for n, result in zip(grids, ou_results):
        print(n, result)
    for method in ['Euler-Maruyama', 'Lie', 'Strang']:
        plt.loglog(1/np.array(grids), [r[method][0] for r in gbm_results], 'o-', label=method)
    plt.xlabel('Step size h')
    plt.ylabel('GBM strong RMSE')
    plt.legend()
    plt.tight_layout()
    plt.savefig('gbm_convergence.png', dpi=180)
    plt.close()


GBM: strong RMSE, weak mean bias
8 {'Euler-Maruyama': (np.float64(0.10855213684646386), np.float64(0.007787305571473002)), 'Lie': (np.float64(2.376762657715184e-16), np.float64(0.0012085193498718727)), 'Strang': (np.float64(7.671555571665091e-16), np.float64(0.0012085193498725388))}
16 {'Euler-Maruyama': (np.float64(0.0729052579114631), np.float64(0.001003112292025654)), 'Lie': (np.float64(8.0597445898156e-16), np.float64(0.003312766242793086)), 'Strang': (np.float64(5.093368884334666e-16), np.float64(0.003312766242794085))}
32 {'Euler-Maruyama': (np.float64(0.05317514716940867), np.float64(0.0017768311400446768)), 'Lie': (np.float64(5.727036949205552e-16), np.float64(0.00042564289260937915)), 'Strang': (np.float64(1.6484960930823767e-15), np.float64(0.00042564289261093347))}
64 {'Euler-Maruyama': (np.float64(0.03657914508706033), np.float64(0.0023321685501880296)), 'Lie': (np.float64(1.6095066785392097e-15), np.float64(0.002997760528100968)), 'Strang': (np.float64(7.229590552395273e-1